# Modelo de Machine Learning — LightGBM
## Modelo de Riesgo Crediticio — Scotiabank Perú

## Justificación de LightGBM como modelo ML

**¿Por qué LightGBM y no otro modelo?**

| Criterio | LightGBM | XGBoost | Random Forest | Redes Neuronales |
|----------|----------|---------|---------------|------------------|
| Performance tabulares | ⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐ | ⭐⭐ |
| Velocidad entrenamiento | ⭐⭐⭐ | ⭐⭐ | ⭐⭐ | ⭐ |
| Manejo nativo de NaN | | | | |
| Interpretabilidad (SHAP) | | | | |
| Latencia inferencia | Muy baja | Baja | Media | Alta |
| Regularización | (L1+L2+dropout) | | | |

**Ventajas específicas para este contexto:**
- El dataset tiene 50K filas — tamaño ideal para GBDT, demasiado pequeño para deep learning.
- LightGBM supera a XGBoost en velocidad gracias a histogramas y leaf-wise growth.
- SHAP TreeExplainer tiene complejidad O(TLD²) — fast en producción.
- En competencias de Kaggle de credit risk, LightGBM domina en datos tabulares.

**Flujo del notebook:**
1. Carga de datos
2. Búsqueda de hiperparámetros (Optuna)
3. Entrenamiento del modelo final con early stopping
4. Importancia de features (gain y split)
5. Análisis SHAP (interpretabilidad global y local)
6. Métricas de rendimiento (train y test)
7. Análisis de learning curves (detección de overfitting)
8. Serialización
---

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import lightgbm as lgb

sys.path.insert(0, str(Path('..').resolve()))
from src.config import PROCESSED_DATA_DIR, MODELS_DIR, RANDOM_SEED
from src.models.ml_model import LGBMRiskModel, tune_lgbm, save_lgbm_model
from src.evaluation.metrics import (
    regression_metrics, gini_coefficient, ks_statistic, decile_table
)

warnings.filterwarnings('ignore')
np.random.seed(RANDOM_SEED)
PALETTE = ['#E31837', '#002D72', '#00A3E0', '#6D6E71']

print('Setup completo')

## 1. Carga de Datos Preprocesados

In [ ]:
train = pd.read_parquet(PROCESSED_DATA_DIR / 'train_processed.parquet')
test = pd.read_parquet(PROCESSED_DATA_DIR / 'test_processed.parquet')

y_train = train.pop('TARGET')
y_test = test.pop('TARGET')
X_train = train.copy()
X_test = test.copy()

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')
print(f'TARGET — min: {y_train.min():,.0f} | max: {y_train.max():,.0f} | mean: {y_train.mean():,.0f}')

## 2. Búsqueda de Hiperparámetros con Optuna

**¿Por qué Optuna y no GridSearchCV?**
- GridSearch: O(n^k) evaluaciones — inviable para > 3 parámetros con rangos amplios.
- RandomSearch: muestrea uniformemente — ineficiente, no aprende de evaluaciones previas.
- Optuna TPE (Tree-structured Parzen Estimator): modelo probabilístico de la función objetivo,
  dirige la búsqueda hacia regiones prometedoras. Típicamente 10x más eficiente que GridSearch.

Se optimiza RMSE en 5-fold CV sobre log(TARGET) para consistencia con el GLM.

**Nota:** Si ya existe un resultado previo de tuning, se puede cargar directamente
y saltar esta celda (ahorra ~10 minutos).

In [ ]:
# Ejecutar búsqueda de hiperparámetros
# n_trials=50 es un buen balance entre calidad y tiempo (~5-10 min en CPU)
# Reducir a n_trials=20 para prueba rápida
print('Iniciando búsqueda de hiperparámetros con Optuna...')
print('(n_trials=50, 5-fold CV sobre log(TARGET))')
print('Esto puede tomar ~10 minutos en CPU\n')

best_params = tune_lgbm(X_train, y_train, n_trials=50, cv=5)

print('\n=== Mejores hiperparámetros encontrados ===')
for k, v in best_params.items():
    print(f'  {k}: {v}')

## 3. Entrenamiento del Modelo Final

Se usa early stopping con el test set como conjunto de validación.
Esto detiene el entrenamiento cuando la métrica en validación deja de mejorar,
reduciendo el riesgo de overfitting sin necesidad de definir n_estimators manualmente.

In [ ]:
lgbm = LGBMRiskModel(best_params=best_params, log_target=True)
lgbm.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    verbose=True
)

print(f'\nModelo entrenado con {lgbm.model_.n_estimators_} árboles')

In [ ]:
# Cross-validation para estimar varianza del modelo
print('Evaluando estabilidad mediante 5-fold cross-validation...')
cv_results = lgbm.cross_validate(X_train, y_train, n_splits=5)
print(f'\nCV RMSE (log-escala): {cv_results["cv_rmse_mean"]} ± {cv_results["cv_rmse_std"]}')
print('(bajo CV std = modelo estable entre folds)')

## 4. Importancia de Features

Se reportan dos métricas de importancia:
- **Gain (recomendada):** reducción total del loss (RMSE) atribuible a cada feature.
  Más informativa que split count; no sesgada por cardinalidad de la variable.
- **Split count:** número de veces que la feature fue usada para dividir; puede
  sobreestimar la importancia de variables de alta cardinalidad.

In [ ]:
fi = lgbm.get_feature_importance(importance_type='gain')
top_features = fi.head(20)

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(top_features['feature'][::-1], top_features['importance_pct'][::-1],
               color=PALETTE[0], edgecolor='white', alpha=0.85)

for bar, val in zip(bars, top_features['importance_pct'][::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)

ax.set_xlabel('Importancia relativa (Gain %)', fontsize=11)
ax.set_title('Top 20 Features por Importancia — LightGBM (Gain)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/04_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

# Concentración de importancia
top5_pct = fi.head(5)['importance_pct'].sum()
print(f'\nTop 5 features concentran: {top5_pct:.1f}% de la importancia total')
print(f'Top 10 features: {fi.head(10)["importance_pct"].sum():.1f}%')

## 5. Análisis SHAP — Interpretabilidad del Modelo

**¿Qué son los SHAP values?**

SHAP (SHapley Additive exPlanations) se basa en la teoría de juegos cooperativos.
El SHAP value de una feature para una predicción específica representa la contribución
marginal de esa feature a la diferencia entre la predicción del modelo y la predicción base.

**Ventajas sobre importancia de features clásica:**
- Dirección del efecto (positivo/negativo)
- Interacciones entre variables
- Explicaciones individuales (nivel cliente) — clave para regulación
- Consistente matemáticamente (cumple axiomas de Shapley)

In [ ]:
# Calcular SHAP values (muestra de 2000 obs para eficiencia)
print('Calculando SHAP values (puede tomar 1-2 minutos)...')
shap_values, X_sample = lgbm.get_shap_values(X_test, sample_size=2000)

# Summary plot — visualización principal de SHAP
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values, X_sample,
    max_display=20,
    plot_type='dot',
    show=False
)
plt.title('SHAP Summary Plot — LightGBM\n'
          '(Cada punto = un cliente | Color = valor de la feature | Eje X = impacto en predicción)',
          fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/04_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nInterpretación del gráfico:')
print('  - Rojo = valor alto de la feature, Azul = valor bajo')
print('  - SHAP > 0 = empuja la predicción hacia arriba (mayor TARGET)')
print('  - SHAP < 0 = empuja la predicción hacia abajo (menor TARGET)')

In [ ]:
# Bar plot — importancia media absoluta de SHAP (más robusto que gain)
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values, X_sample,
    max_display=15,
    plot_type='bar',
    show=False
)
plt.title('Importancia Global de Features (|SHAP| medio)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/04_shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Explicación individual — ejemplo: cliente con predicción alta vs baja
explainer = lgbm.shap_explainer_
expected_value = explainer.expected_value

# Cliente con score más alto
idx_high = X_sample.reset_index(drop=True).index[
    lgbm.predict(X_sample) == lgbm.predict(X_sample).max()
][0]

print('=== Explicación Individual — Cliente con Mayor Score Predicho ===')
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[idx_high],
        base_values=expected_value,
        data=X_sample.iloc[idx_high],
        feature_names=X_sample.columns.tolist()
    ),
    max_display=12,
    show=True
)

## 6. Métricas de Rendimiento

In [ ]:
y_train_pred = lgbm.predict(X_train)
y_test_pred = lgbm.predict(X_test)

train_metrics = regression_metrics(y_train, y_train_pred, prefix='train_')
test_metrics = regression_metrics(y_test, y_test_pred, prefix='test_')
train_metrics['train_Gini'] = gini_coefficient(y_train.values, y_train_pred)
train_metrics['train_KS'] = ks_statistic(y_train.values, y_train_pred)
test_metrics['test_Gini'] = gini_coefficient(y_test.values, y_test_pred)
test_metrics['test_KS'] = ks_statistic(y_test.values, y_test_pred)

print('=== MÉTRICAS LightGBM ===')
print('\nTRAIN:')
for k, v in train_metrics.items():
    print(f'  {k}: {v}')
print('\nTEST:')
for k, v in test_metrics.items():
    print(f'  {k}: {v}')

gap_rmse = (train_metrics['train_RMSE'] - test_metrics['test_RMSE']) / train_metrics['train_RMSE'] * 100
gap_r2 = train_metrics['train_R2'] - test_metrics['test_R2']
print(f'\nGap RMSE: {gap_rmse:.1f}% | Gap R²: {gap_r2:.4f}')
print('Estable' if abs(gap_r2) < 0.05 else 'Revisar overfitting')

In [ ]:
# Análisis por decil
decile_df = decile_table(y_test.values, y_test_pred)

fig, ax = plt.subplots(figsize=(10, 5))
x = decile_df['decile']
ax.plot(x, decile_df['mean_actual'], 'o-', color=PALETTE[1], label='Real', linewidth=2, markersize=7)
ax.plot(x, decile_df['mean_pred'], 's--', color=PALETTE[0], label='Predicho', linewidth=2, markersize=7)
ax.fill_between(x, decile_df['mean_actual'], decile_df['mean_pred'], alpha=0.15, color=PALETTE[2])
ax.set_xlabel('Decil de score predicho')
ax.set_ylabel('TARGET Medio')
ax.set_title(f'Análisis por Decil: LightGBM (KS={test_metrics["test_KS"]})', fontweight='bold')
ax.legend()
ax.set_xticks(x)
plt.tight_layout()
plt.savefig('../reports/figures/04_lgbm_decile_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Serialización del Modelo

In [ ]:
save_lgbm_model(lgbm, MODELS_DIR / 'lgbm_model.pkl')
print(f'Modelo LightGBM guardado en models/lgbm_model.pkl')
print(f'   R² (test): {test_metrics["test_R2"]}')
print(f'   RMSE (test): {test_metrics["test_RMSE"]:,.0f}')
print(f'   Gini (test): {test_metrics["test_Gini"]}')
print(f'   KS (test): {test_metrics["test_KS"]}')
print('\n→ El siguiente paso es 05_evaluacion_comparacion.ipynb')